# Relative Effectiveness
Generate habitat condition and loss metrics within each cell, and calculate relative effectiveness metrics by comparing scores for matched treatment and control cells.

In [ ]:
# Select site by WDPAID (one from the list of 30 in variables.py)
site_id = 916

In [ ]:
from pathlib import Path
import os
import sys
import ee
import geemap
import geopandas as gpd

cur = Path.cwd().resolve()
for parent in [cur] + list(cur.parents):
    if parent.name == "tpae":
        os.chdir(parent)
        break

sys.path.insert(0, str((Path.cwd() / "src").resolve()))

from utils.variables import (
    PROJECT,
    ANALYSIS_END_YR,
)

from absolute_effectiveness.site_selector import SiteSelector
from absolute_effectiveness.data_processor import DataProcessor
from relative_effectiveness.metrics_per_cell import (
    RelativeHabitatConditionAnalyzer,
    RelativeHabitatLossAnalyzer,
)
from relative_effectiveness.relative_scores import (
    load_match_table,
    build_match_df_with_score_diffs,
    aggregate_pa_relative_scores,
)

ee.Authenticate()
ee.Initialize(project=PROJECT)

site_selector = SiteSelector()
processor = DataProcessor.from_gee_defaults()
condition_analyzer = RelativeHabitatConditionAnalyzer()
loss_analyzer = RelativeHabitatLossAnalyzer()

In [ ]:
# Load matched_grids and derive site-specific context

matched_grids_gdf = gpd.read_parquet(f"data/mdm/matched_grids_mdm_{site_id}.parquet").to_crs(epsg=4326)
matched_grids = geemap.geopandas_to_ee(matched_grids_gdf)

test_sites = site_selector.get_test_sites()
START_YR = site_selector.set_start_yr(test_sites, site_id)
site_geom = site_selector.get_site_geom(test_sites, site_id)

site_selector.check_start_yr(START_YR)

In [ ]:
# Process input datasets
GLC_processed = processor.process_glc(matched_grids, START_YR)
GPW_processed = processor.process_gpw(START_YR)
NFW_processed = processor.process_nfw(matched_grids)
HGFC_processed = processor.process_hgfc(START_YR)
land_mask = processor.get_land_mask()

# Build land-masked habitat raster for extent scoring
habitat_raster = condition_analyzer.get_habitat_raster(
    GLC_processed, HGFC_processed, GPW_processed, NFW_processed
)

# Build unmasked habitat raster for intactness (water excluded inside kernel, not zeroed)
GLC_intactness = processor.process_glc(matched_grids, START_YR, land_masked=False)
GPW_intactness = processor.process_gpw(START_YR, land_masked=False)
NFW_intactness = processor.process_nfw(matched_grids, land_masked=False)
HGFC_intactness = processor.process_hgfc(START_YR, land_masked=False)
habitat_raster_intactness = condition_analyzer.get_habitat_raster(
    GLC_intactness, HGFC_intactness, GPW_intactness, NFW_intactness
)

exp_kernel = condition_analyzer.build_kernel()
intactness_raster = condition_analyzer.get_intactness_raster(
    habitat_raster_intactness, matched_grids.geometry(), exp_kernel, land_mask
)

scored = condition_analyzer.calc_extent_score_per_cell(habitat_raster, matched_grids)
scored = condition_analyzer.calc_intactness_score_per_cell(intactness_raster, scored)
scored = condition_analyzer.calc_condition_score_per_cell(scored)

# Build habitat-loss raster and score Habitat Loss per cell
habitat_loss_raster, _ = loss_analyzer.get_habitat_loss_raster(
    GLC_processed, GPW_processed, HGFC_processed, START_YR
)
scored = loss_analyzer.calc_loss_score_per_cell(habitat_loss_raster, scored)


# print(f"Site ID: {site_id}")
# print(f"Analysis Period: {START_YR} - {ANALYSIS_END_YR}")
# print(f"Scored cells: {scored.size().getInfo()}")
# print("\nFirst feature properties:")
# print(scored.first().getInfo()["properties"])

## Calculate relative effectiveness
Calculate differences in effectiveness metrics between matched treatment and control cells, and aggregate relative metrics for the PA.

In [ ]:
match_table = load_match_table(site_id)
match_df = build_match_df_with_score_diffs(scored, match_table)
match_df[['treat_cell_id', 'control_cell_id', 'treat_condition_score', 'control_condition_score', 'condition_score_diff', 'treat_loss_score', 'control_loss_score', 'loss_score_diff']].head()

In [ ]:
treat_means, pa_scores = aggregate_pa_relative_scores(match_df)

print("PA Relative Extent Score: {:.2f}".format(pa_scores["extent"]))
print("PA Relative Intactness Score: {:.2f}".format(pa_scores["intactness"]))
print("PA Relative Condition Score: {:.2f}".format(pa_scores["condition"]))
print("PA Relative Loss Score: {:.2f}".format(pa_scores["loss"]))

## Visualization

In [ ]:
from utils.variables import GLC_PALETTE

score_palette = ["#d73027", "#fdae61", "#fee08b", "#d9ef8b", "#1a9850"]
score_viz = {"min": 0, "max": 1, "palette": score_palette}

def score_image(fc, prop):
    """Rasterize a numeric per-feature score so it can be displayed with a palette."""
    return fc.reduceToImage(properties=[prop], reducer=ee.Reducer.first())

Map = geemap.Map(height=900)
Map.add_basemap("CartoDB.DarkMatter")
Map.add_basemap("Esri.WorldImagery")

Map.addLayer(habitat_raster, {"palette": GLC_PALETTE}, "Habitat Extent", 0)
Map.addLayer(intactness_raster, {"min": 0, "max": 1, "palette": ["#d73027", "#fdae61", "#fee08b", "#d9ef8b", "#1a9850"]}, "Habitat Intactness", 0)

Map.addLayer(site_geom, {"color": "white"}, "Test site", 1, 0.5)
# Map.addLayer(matched_grids, {"color": "white"}, "Matched cells (outline)", False)

Map.addLayer(score_image(scored, "extent_score"), score_viz, "Extent Score", 0)
Map.addLayer(score_image(scored, "intactness_score"), score_viz, "Intactness Score", 0)
Map.addLayer(score_image(scored, "condition_score"), score_viz, "Condition Score")
Map.addLayer(score_image(scored, "loss_score"), score_viz, "Loss Score")

Map.centerObject(matched_grids)

Map